# A_15 — "Fatores de risco" no SIVEP: as comorbidades em branco são "não"?

**Hipótese.** A ficha tem a pergunta-mãe `fator_risc` ("possui fatores de risco/comorbidades?"). Se a lista de comorbidades só é preenchida quando a resposta é "sim", o branco numa comorbidade com `fator_risc` = não significa "não tem", e não "não informado".

**Confirma a hipótese** se, com `fator_risc` = não, o branco for quase 100% e, com `fator_risc` = sim, bem menor (tabela 1).

Este notebook só lê `coorte_final.parquet` (colunas necessárias) e **não treina nada**. Comente as três tabelas no card com a proposta de regra.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

RAIZ = Path.cwd()
while not (RAIZ / "configs").exists() and RAIZ != RAIZ.parent:   # AJUSTAR se o projeto tiver outro jeito de achar a raiz
    RAIZ = RAIZ.parent
COORTE = RAIZ / "data/analytical/coorte_final.parquet"
SAIDA = RAIZ / "outputs/tables"

COMORB = ["cardiopati", "hematologi", "sind_down", "hepatica", "asma", "diabetes",
          "neurologic", "pneumopati", "imunodepre", "renal", "obesidade", "out_morbi", "puerpera"]
COLUNAS = ["fator_risc", "ano_internacao", "alvo_obito_srag"] + COMORB
ANOS_REF = [2023, 2024, 2025]          # anos em que o card cita 61% a 69% de branco


def classificar_fator(s: pd.Series) -> pd.Series:
    """sim / nao / ignorado / branco. CONFIRMAR o mapeamento com o value_counts da célula anterior."""
    v = s.astype("string").str.strip().str.upper()
    sim = v.isin(["1", "1.0", "S", "SIM"])
    nao = v.isin(["2", "2.0", "N", "NAO", "NÃO"])
    ign = v.isin(["9", "9.0", "IGNORADO"])
    return pd.Series(np.select([sim, nao, ign], ["sim", "nao", "ignorado"], default="branco"), index=s.index)


def em_branco(df: pd.DataFrame, cols) -> pd.DataFrame:
    """Branco = vazio ou nulo. O código 9 (ignorado) NÃO conta como branco."""
    return df[cols].isna() | (df[cols].astype("string").apply(lambda c: c.str.strip()) == "")


def tabela1_fator_x_branco(d: pd.DataFrame) -> pd.DataFrame:
    """% de branco de cada comorbidade dentro de cada valor de fator_risc, por ano."""
    b = em_branco(d, COMORB)
    b["ano"], b["fator"] = d["ano_internacao"].values, d["grupo_fator"].values
    long = b.melt(id_vars=["ano", "fator"], var_name="comorbidade", value_name="branco")
    t = (long.groupby(["comorbidade", "ano", "fator"])["branco"].agg(n="size", pct_branco=lambda s: s.mean() * 100)
         .round(2).reset_index())
    return t


def tabela2_mortalidade(d: pd.DataFrame) -> pd.DataFrame:
    t = (d.groupby(["ano_internacao", "grupo_fator"])["alvo_obito_srag"]
         .agg(n="size", obitos="sum").reset_index())
    t["mortalidade_pct"] = (t["obitos"] / t["n"] * 100).round(2)
    return t


def tabela3_ausencia_por_desfecho(d: pd.DataFrame, anos=ANOS_REF) -> pd.DataFrame:
    """% de branco em curas e óbitos (diferença em pontos percentuais): todas as fichas x só fator_risc = sim."""
    saidas = []
    for rotulo, sub in (("todas", d), ("so_fator_sim", d[d["grupo_fator"] == "sim"])):
        sub = sub[sub["ano_internacao"].isin(anos)]
        b = em_branco(sub, COMORB)
        for c in COMORB:
            cura = b.loc[sub["alvo_obito_srag"] == 0, c].mean() * 100
            obito = b.loc[sub["alvo_obito_srag"] == 1, c].mean() * 100
            saidas.append((rotulo, c, int(len(sub)), round(cura, 2), round(obito, 2), round(cura - obito, 2)))
    return pd.DataFrame(saidas, columns=["populacao", "comorbidade", "n", "pct_branco_cura", "pct_branco_obito", "dif_pp_cura_menos_obito"])


In [ ]:
d = pd.read_parquet(COORTE, columns=COLUNAS)
print(len(d), "fichas")
# 1) valores de fator_risc (podem variar por ano: 1/2/9, S/N ou texto). Anote o mapeamento.
print(d.groupby("ano_internacao")["fator_risc"].value_counts(dropna=False).unstack(fill_value=0))
d["grupo_fator"] = classificar_fator(d["fator_risc"])
print(d["grupo_fator"].value_counts())

In [ ]:
t1 = tabela1_fator_x_branco(d)
# resumo de leitura: % de branco médio nos anos de referência, por valor de fator_risc
resumo = (t1[t1["ano"].isin(ANOS_REF)].pivot_table(index="comorbidade", columns="fator", values="pct_branco", aggfunc="mean").round(1))
print(resumo)

In [ ]:
t2 = tabela2_mortalidade(d)
print(t2[t2["ano_internacao"].isin(ANOS_REF)])

In [ ]:
t3 = tabela3_ausencia_por_desfecho(d)
print(t3.pivot_table(index="comorbidade", columns="populacao", values="dif_pp_cura_menos_obito"))
# Se a diferença de 13 a 21 pp cair muito em so_fator_sim, parte do "viés de documentação" era a regra da ficha.

In [ ]:
SAIDA.mkdir(parents=True, exist_ok=True)
t1.to_csv(SAIDA / "eda_02_fator_risc_comorbidades.csv", index=False)
t2.to_csv(SAIDA / "eda_02_fator_risc_mortalidade.csv", index=False)
t3.to_csv(SAIDA / "eda_02_fator_risc_ausencia_desfecho.csv", index=False)